# 实验一：静态页面与 API 数据采集

**平台：SmartPark DataLab（`https://115.159.159.82`）**

本实验完成两类典型 Web 数据采集：

| 数据源 | 加载方式 | 采集路径 | 目标 |
|---|---|---|---|
| Device（设备资产） | 动态加载 | 找 API → JSON → 分页 | 5000 条 |
| Company（园区企业） | 静态加载 | HTML → BeautifulSoup → 下一页 | 3 页 60 条 |

> 运行方式：JupyterLab 中 `Restart Kernel → Run All`，最后一个单元格应输出
> `Company 总数 60 / Device 总数 5000 / 不一致企业数 0`。

## 0. 环境准备

只需要 `requests`（发请求）和 `beautifulsoup4`（解析 HTML），课程容器中已经装好。

- `BASE_URL` 是平台根地址，后面所有页面、API 都在它下面，这样写方便统一修改。

In [1]:
import json
from collections import Counter
from pathlib import Path
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://115.159.159.82"

print("环境准备完成")

环境准备完成


---

# 任务一：Device API 分页采集

## 1.1 判断加载方式并找到 API

**第一步：右键 → 查看网页源代码**，按 `Ctrl + F` 搜索页面上真实显示的设备名称（例如 `循环泵-000001-001`）。

结果：**搜不到**。说明设备数据不是跟着 HTML 一起返回的，页面是**动态加载**。

**第二步：F12 → Network → Fetch/XHR**，刷新 `/devices`，找到返回设备列表的请求：

```text
Request URL : https://115.159.159.82/api/devices?page=1&page_size=50
Method      : GET
Query 参数   : page、page_size（还有可选的 company_id / device_type / status / zone）
```

## 1.2 用 requests 复现第一页请求

按 Network 中观察到的地址和参数，把请求复现出来。

In [2]:
api_url = BASE_URL + "/api/devices"

r = requests.get(
    api_url,
    params={
        "page": 1,        # 当前页
        "page_size": 50,  # 每页条数（与浏览器里观察到的一致）
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json


`status == 200` 只说明“服务器把东西返回来了”，**并不代表数据是对的**，后面任务四还要做真正的校验。

## 1.3 读 JSON，确认分页字段

In [3]:
data = r.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [4]:
# 分页结构：这四个字段分别代表什么？
print("当前页   page      =", data["page"])
print("每页条数 page_size =", data["page_size"])
print("设备总数 total     =", data["total"])
print("本页列表 items     =", len(data["items"]), "条")

# 看一条真实设备记录
print()
print(data["items"][0])

当前页   page      = 1
每页条数 page_size = 50
设备总数 total     = 5000
本页列表 items     = 50 条

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


设备记录字段说明：

| 字段 | 含义 |
|---|---|
| `device_id` | 设备编号，如 `D00000001` |
| `company_id` | 所属企业编号，如 `C000001` |
| `device_name` | 设备名称 |
| `device_type` | 类型（pump / compressor / fan / motor / heat_exchanger / storage_tank） |
| `model` | 型号 |
| `zone` | 所在区域 |
| `rated_power_kw` | 额定功率（kW） |
| `status` | 运行状态（running / stopped / maintenance / fault） |

## 1.4 自动计算总页数并采集全部设备

**总页数不要写死**，用接口返回的 `total` 和 `page_size` 算出来。
这样当平台数据量变化时（5000 变 8000），代码不用改也能跑。

In [5]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size  # 向上取整

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [6]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size,
        },
        timeout=10
    )

    page_data = r.json()

    # 把本页设备追加进 devices
    devices.extend(page_data["items"])

print("device count:", len(devices))
print("unique device_id:", len(set(x["device_id"] for x in devices)))

device count: 5000
unique device_id: 5000


---

# 任务二：Company 静态页面解析

## 2.1 判断加载方式

打开 `/companies`，**右键 → 查看网页源代码**，按 `Ctrl + F` 搜索页面上的企业名称（例如 `临港精工001有限公司`）。

结果：**能搜到**。说明企业数据已经写在 HTML 里了，属于**静态加载**，可以直接请求 HTML 再用 BeautifulSoup 解析。

In [7]:
current_url = BASE_URL + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

print("status:", r.status_code)
print("html 长度:", len(r.text))

status: 200
html 长度: 11331


## 2.2 设计企业记录的 CSS Selector

在 Elements 中看一条企业记录的外层节点：

```html
<article class="company-card" data-company-id="C000001">
  <div class="company-card-top">
    <span class="mono">C000001</span>
    <span class="badge active">运营中</span>
  </div>
  <h2><a href="/companies/C000001">临港精工001有限公司</a></h2>
  <dl>
    <div><dt>所属行业</dt><dd>汽车零部件</dd></div>
    <div><dt>所在区域</dt><dd>智造一区</dd></div>
    <div><dt>员工规模</dt><dd>206 人</dd></div>
    <div><dt>设备数量</dt><dd>6 台</dd></div>
  </dl>
  <a class="detail-link" href="/companies/C000001">查看企业档案 →</a>
</article>
```

按“优先级顺序”考虑：

1. 有没有语义明确的 `id`？→ 没有；
2. 有没有稳定的 `class`？→ **有**：`company-card` 只用于企业卡片，语义明确；
3. 有没有 `data-*`？→ 有 `data-company-id`，用来取企业编号；
4. 父子层级？→ 不需要。

所以最终 selector 就用最短、最语义化的一层：

```text
article.company-card
```

> 浏览器 Copy selector 给出的可能是很长的 `nth-child(...)` 路径，那种写法一旦页面加了一条记录就会失效。

In [8]:
# 一条企业记录的最外层节点就是 article.company-card
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


## 2.3 提取企业字段

**先定位一条记录，再在记录内部提取字段。**

`company_id`、`name`、`status` 直接从当前记录内部取；
行业 / 区域 / 设备数量是 `dl > div > dt + dd` 结构，先转成字典再取值。

In [9]:
card = cards[0]

# ① dt/dd 结构先转成字典
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

# ② 其他字段在记录内部定位
company_id = card.get("data-company-id")          # 来自 HTML 属性 data-company-id
name = card.select_one("h2 a").get_text(strip=True)  # 来自 <h2><a> 文本
status = card.select_one(".badge").get_text(strip=True)  # 来自 .badge 文本

print()
print("company_id:", company_id)
print("name:", name)
print("status:", status)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}

company_id: C000001
name: 临港精工001有限公司
status: 运营中


In [10]:
# ③ 行业 / 区域 / 设备数量
industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())  # "6 台" → 6

print("industry:", industry)
print("zone:", zone)
print("device_count:", device_count)

industry: 汽车零部件
zone: 智造一区
device_count: 6


把上面的提取过程封装成函数，方便对每一条记录复用：

In [11]:
def parse_card(card):
    """把一条 article.company-card 解析成字典"""
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card.get("data-company-id"),
        "name": card.select_one("h2 a").get_text(strip=True),
        "industry": details.get("所属行业"),
        "zone": details.get("所在区域"),
        "device_count": int(details.get("设备数量", "0").replace("台", "").strip()),
        "status": card.select_one(".badge").get_text(strip=True),
    }


page1_companies = [parse_card(c) for c in cards]

print("本页企业数：", len(page1_companies))
print(page1_companies[0])

本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


---

# 任务三：根据“下一页”连续采集 3 页

页面底部的分页结构：

```html
<!-- 第 1 页 -->
<nav class="pagination"><span>上一页</span><b>1</b><a href="/companies?page=2">下一页</a></nav>

<!-- 第 2 页 -->
<nav class="pagination"><a href="/companies?page=1">上一页</a><b>2</b><a href="/companies?page=3">下一页</a></nav>
```

注意第 1 页的“上一页”是 `<span>`，第 2 页起才变成 `<a>`。
所以**不能**简单写 `nav.pagination a`（第 2 页会取到“上一页”），要用 `a:last-of-type` 取最后一个链接 —— 也就是“下一页”。

In [12]:
next_link = soup.select_one("nav.pagination a:last-of-type")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

# href 是相对路径，需要拼成完整地址
next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

下一页
/companies?page=2
https://115.159.159.82/companies?page=2


## 3.1 循环：当前页 → 找下一页 → 请求下一页 → 再解析

把“请求当前页 / 解析本页企业 / 找下一页 href”三步放进一个 `for` 循环，
每轮结束后把 `current_url` 换成刚找到的下一页地址，就能一页一页往下走。

> 第 1 页已经请求过了，这里为了代码完整，循环 3 次、每次都用 `current_url` 重新请求。

In [13]:
companies = []
page_counts = []

for i in range(3):  # 连续采集 3 页
    r = requests.get(current_url, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    page_cards = soup.select("article.company-card")
    page_counts.append(len(page_cards))

    for card in page_cards:
        companies.append(parse_card(card))

    # 找“下一页”
    next_link = soup.select_one("nav.pagination a:last-of-type")
    if next_link is None:
        break

    current_url = urljoin(current_url, next_link.get("href"))

print("每页企业数：", page_counts)
print("Company 总数：", len(companies))

每页企业数： [20, 20, 20]
Company 总数： 60


---

# 任务四：验证采集结果

最后把 5 个验证数字放在**同一个单元格**里打印，方便一次性截图（`images/04_validation.png`）。

In [14]:
print("=== 最终验证结果 ===")
print("Company 总数        :", len(companies))
print("唯一 company_id 数  :", len(set(x["company_id"] for x in companies)))
print("Device 总数         :", len(devices))
print("唯一 device_id 数   :", len(set(x["device_id"] for x in devices)))

# Company–Device 一致性：用全部 Device 按 company_id 统计实际设备数
device_counts = Counter(x["company_id"] for x in devices)

mismatch_count = 0

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]      # 企业页面上写的设备数量
    actual = device_counts.get(cid, 0)      # 设备表里真实出现的条数

    if expected != actual:
        mismatch_count += 1

print("不一致企业数        :", mismatch_count)

=== 最终验证结果 ===
Company 总数        : 60
唯一 company_id 数  : 60
Device 总数         : 5000
唯一 device_id 数   : 5000
不一致企业数        : 0


**为什么 `status_code == 200` 还不够？**

返回 200 只说明服务器正常响应了，数据完全可能是空的、重复的、或者只拿到了第 1 页。
所以必须再检查：条数对不对、`id` 有没有重复、两个数据源之间能不能对上。
上面的 `mismatch_count` 就是拿企业页面的 `device_count` 和真实设备记录做交叉验证。

---

# 任务五：保存采集结果

本实验只用 JSON 保存，暂不引入 Pandas、数据库或 Parquet。

In [15]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")
print(sorted(p.name for p in data_dir.iterdir()))

数据已保存到 data/ 目录
['companies.json', 'devices.json']


---

# 完成前自检

```text
☑ 会用“查看网页源代码”判断静态 / 动态加载
☑ 能在 Network 中找到 Device API
☑ 能读懂 API 分页结构并自动计算总页数
☑ Device 共采集 5000 条，device_id 无重复
☑ 能设计简洁、稳定的 Company CSS Selector
☑ 能定位“下一页”并读取 href
☑ Company 连续采集 3 页，共 60 条
☑ company_id 无重复
☑ Company–Device 数量检查通过（不一致 0 家）
☑ 已保存 companies.json 和 devices.json
```